In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

from pathlib import Path
import pickle

from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import equinox as eqx
import jax
import jax.numpy as jnp

In [29]:
seq_len = 90
n_nodes = 7000
hidden_size = 32
targets = ['discharge']

key = jax.random.PRNGKey(0)
state = jax.random.uniform(key, (seq_len, n_nodes, hidden_size))
state.shape

(90, 7000, 32)

In [57]:
from jaxtyping import Array, PRNGKeyArray

class GMM(eqx.Module):
    fc1: eqx.nn.Linear
    fc2: eqx.nn.Linear
    _eps: float

    def __init__(self, in_size: int, out_size: int, hidden_size: int, *, key: PRNGKeyArray):
        k1, k2 = jax.random.split(key)
        self.fc1 = eqx.nn.Linear(in_size, hidden_size, key=k1)
        self.fc2 = eqx.nn.Linear(hidden_size, out_size * 3, key=k2)
        self._eps = 1e-5

    def __call__(self, x: Array) -> dict[str, Array]:
        """Perform a GMM head forward pass.

        Parameters
        ----------
        x : Array
            Final latent variables to compute the GMM components.

        Returns
        -------
        dict[str, Array]
            Dictionary containing mixture parameters and weights; where the key 'mu' stores the means, the key
            'sigma' the variances, and the key 'pi' the weights.
        """
        h = jax.nn.relu(self.fc1(x))
        h = self.fc2(h)

        # split output into mu, sigma and weights
        mu, s_latent, p_latent = jnp.split(h, 3, axis=-1)
        sigma = jnp.exp(s_latent) + self._eps
        pi = jax.nn.softmax(p_latent, axis=-1)

        return {'mu': mu, 'sigma': sigma, 'pi': pi}

n_dist_models = 100

heads = {
    target: GMM(hidden_size, n_dist_models, hidden_size * 2, key=key)
    for target in targets
}

heads

{'discharge': GMM(
   fc1=Linear(
     weight=f32[64,32],
     bias=f32[64],
     in_features=32,
     out_features=64,
     use_bias=True
   ),
   fc2=Linear(
     weight=f32[300,64],
     bias=f32[300],
     in_features=64,
     out_features=300,
     use_bias=True
   ),
   _eps=1e-05
 )}

In [31]:
last_state = state[-1,...]
last_state.shape

(7000, 32)

In [43]:
node_mask = jnp.ones((n_nodes,))

(7000,)

In [89]:
#vmap each target over each location
y_pred = {
    target: jax.vmap(head)(last_state)
    for target, head in heads.items()
}

In [66]:
out['discharge']['mu'].shape

(7000, 100)

In [75]:
y = jax.random.uniform(key, (n_nodes))
y.shape

(7000,)

In [76]:
m.shape

(7000, 100)

In [87]:
ONE_OVER_2PI_SQUARED = 1.0 / jnp.sqrt(2.0 * jnp.pi)

result = (y[:, None] - m) * jnp.reciprocal(s)
results = -0.5 * result**2

jnp.exp(result) * jnp.reciprocal(s) * ONE_OVER_2PI_SQUARED

Array([[0.9268029 , 0.7977337 , 1.2160528 , ..., 1.926294  , 1.8654932 ,
        1.158916  ],
       [0.91213715, 0.9333715 , 1.425512  , ..., 2.1084714 , 1.6798136 ,
        1.250756  ],
       [0.50777465, 0.5197229 , 0.6796516 , ..., 1.0944709 , 1.1194712 ,
        0.57518   ],
       ...,
       [0.5614755 , 0.6776856 , 0.817822  , ..., 1.1740613 , 1.1810212 ,
        0.698774  ],
       [0.4521821 , 0.4801465 , 0.6721319 , ..., 1.127818  , 1.0035329 ,
        0.53565353],
       [0.48312086, 0.5041933 , 0.6480659 , ..., 0.81879765, 1.0362604 ,
        0.48482034]], dtype=float32)

In [ ]:
jnp.expand_dims(y, 

In [72]:
# def GMM_loss(self, y: Array, y_hat: dict[str, Array], mask: Array):
"""Average negative log-likelihood for a gaussian mixture model (GMM). """
m = y_hat['discharge']['mu']
s = y_hat['discharge']['sigma']
p = y_hat['discharge']['pi']



    # result = _gaussian_distribution(m, s, y) * p
    # result = torch.sum(result, dim=-1)
    # result = -torch.log(result + self.eps)  # epsilon stability
    # return torch.mean(result)


In [90]:
y_hat = y_pred['discharge']

In [ ]:
def gmm_nll_loss(y: Array, y_hat: dict[str, Array], mask: Array) -> Array:
    """
    Calculates the average negative log-likelihood for a Gaussian Mixture Model (GMM).
    """    
    y_unsqueezed = jnp.expand_dims(y, axis=-1) # Allow broadcasting to each error model

    # Calculate the probability density of y for each Gaussian component.
    one_over_sqrt_2pi = 1.0 / jnp.sqrt(2.0 * jnp.pi)
    exponent = -0.5 * jnp.square((y_unsqueezed - y_hat['mu']) / y_hat['sigma'])
    pdf_values = one_over_sqrt_2pi * jnp.exp(exponent) / y_hat['sigma']

    # Weight the PDFs by the mixture weights (pi) and sum them up
    weighted_likelihoods = jnp.sum(y_hat['pi'] * pdf_values, axis=-1)

    # negative log likelihood
    nll = -jnp.log(weighted_likelihoods + 1E-6)
    
    mean_nll = jnp.mean(nll, where=mask)